## Part 2: Go deeper

In [2]:
import pandas as pd

listings  = pd.read_csv("data/listings_clean.csv", parse_dates=["last_scraped", "calendar_last_scraped", "first_review", "last_review"])

print(listings.shape)

(4574, 73)


### 1.

- As the mean is lower than the median, the tail is towards the min value rather than towards the max value as was the case with prices.
- We can also see that it is quite a large formation close to the top boundary(5) as the 25th procentile is at 4.75.
- We also get an indication of something looking like a tail with the 1.00 min value (although we dont know exactly how many values that makeup that tail).
- The type of numbers we are talking about matters as a prices have a "natural" negative boundary at 0 and endless top whearas reviews are usually bound between 1 and 5.
- There is a possibility (based on behaivoral science) that reviewers that are not happy with their accommodations are less likley to leave a review.
****
- As for choosing a value between mean and median I think that it would be good to present both the mean and the median as they are quite close which would indicate quite a symetrical distibution.
- With the abvous caviat that is the low left-tail.
****
- But if we need to choose one value, median is probably the best representations since we can see that so many values are clustered at the top end of the spectrum.

In [4]:
listings["review_scores_rating"].describe().round(2)

count    3930.00
mean        4.82
std         0.32
min         1.00
25%         4.75
50%         4.92
75%         5.00
max         5.00
Name: review_scores_rating, dtype: float64

### 2.

- The lower fence is 4.375 and there are 211 listings below it.
- The median 'number_of_reviews' for the outlier listings is exactly 4, exactly 13 for the others.
- 28.1% of the listings that are not outliers has 5 reviews or fewer, compared to 51,2% for the outliers.
****
- Regarding the outliers it think it would be unwise to remove them since it would undermine the integrity of the data.
- They are outliers in a statistical sense but they are as representative of the total as the positive reviews on aggregate.
- We cannot remove reviews just because it doesnt fit our overall picture, they are not outliers in that sense (not to be compared with a faulty sensor reading for example).
- They are already marked, so we keep them at that to make it easier to dig in to differences between the positive reviews.
****
- A small note: The outliers COULD of course be "false readings" in a number of ways, but we cannot tell if they are, and the 5s could be faulty in the same way. Visitor just clicks on something to get to the next page on the site for example.

In [13]:
q1 = listings["review_scores_rating"].quantile(0.25)
q3 = listings["review_scores_rating"].quantile(0.75)

iqr = q3 - q1

upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr

outliers = listings[listings["review_scores_rating"] < lower]

print(lower, upper, iqr)
print(len(outliers))
outliers["number_of_reviews"].describe()

4.375 5.375 0.25
211


count    211.000000
mean      11.815166
std       26.781816
min        1.000000
25%        2.000000
50%        4.000000
75%       10.000000
max      258.000000
Name: number_of_reviews, dtype: float64

In [18]:
listings["review_score_outlier"] = listings["review_scores_rating"] < lower

print(listings["review_score_outlier"].sum())

211


In [23]:
listings["number_of_reviews"].describe()

count    4574.000000
mean       35.754045
std        78.377200
min         0.000000
25%         2.000000
50%         9.000000
75%        29.000000
max      1293.000000
Name: number_of_reviews, dtype: float64

In [26]:
listings["has_reviews"] = listings["number_of_reviews"] > 0

listings["has_reviews"].value_counts()

has_reviews
True     3930
False     644
Name: count, dtype: int64

In [30]:
reviewed = listings["has_reviews"]

listings.loc[reviewed & ~listings["review_score_outlier"], "number_of_reviews"].agg(["count", "mean", "median", "std"])

count     3719.000000
mean        43.303576
median      13.000000
std         84.877382
Name: number_of_reviews, dtype: float64

In [39]:
less_than_5 = listings["number_of_reviews"] <= 5

(
    listings.loc[less_than_5 & reviewed & ~listings["review_score_outlier"], "number_of_reviews"].count() / 
    listings.loc[reviewed & ~listings["review_score_outlier"], "number_of_reviews"].count()
    )

np.float64(0.28125840279645065)

In [ ]:
(
    listings.loc[less_than_5 & reviewed & listings["review_score_outlier"], "number_of_reviews"].count() / 
    listings.loc[reviewed & listings["review_score_outlier"], "number_of_reviews"].count()
    )

np.float64(0.5924170616113744)

### 3.

a)

In [41]:
listings["host_is_superhost"].value_counts()

host_is_superhost
False    3837
True      737
Name: count, dtype: int64

In [42]:


super_vs_other = pd.DataFrame({
    "other": listings.loc[~listings["host_is_superhost"], "number_of_reviews"].agg(["count", "median"]),
    "superhosts" : listings.loc[listings["host_is_superhost"], "number_of_reviews"].agg(["count", "median"]),
}).round(0)

super_vs_other

,other,superhosts
count,3837.0,737.0
median,7.0,53.0


b)
- I do not think that median age of listing for the two groups explains much. 
- Especially when concidering the huge spread of number of years in general.

In [46]:
print(listings["last_scraped"].dtype)
print(listings["first_review"].dtype)

datetime64[us]
datetime64[us]


In [49]:
listings["years_existed"] = (listings["last_scraped"] - listings["first_review"]).dt.days / 365.25

listings["years_existed"].describe()

count    3930.000000
mean        3.886026
std         3.403565
min         0.000000
25%         1.190965
50%         2.988364
75%         5.945243
max        15.162218
Name: years_existed, dtype: float64

In [55]:
super_vs_other = pd.DataFrame({
    "other": listings.loc[~listings["host_is_superhost"], "years_existed"].agg(["mean", "median", "std"]),
    "superhosts" : listings.loc[listings["host_is_superhost"], "years_existed"].agg(["mean", "median", "std"]),
}).round(2)

super_vs_other

,other,superhosts
mean,3.79,4.31
median,2.89,3.31
std,3.39,3.45


c)
- Here we can see that superhosts on aggregate have a lot more reviews in the last year, as they do in general.
- But we cannot say is the cause of it. Or the causal direction, because: we do not know if being a superhost renders more reviews or if many reviews gives superhost status.
****
- Upon looking up the criteria for beeing a superhost there is a requirement for completed reservations that could render more reviews, more guests = more reviews on avg. This could be part of the reason.
- There are also some quality criteria in there, and if we agree that happy customers are more likley to leave a review this could be a contributing factor.

In [57]:
super_vs_other = pd.DataFrame({
    "other": listings.loc[~listings["host_is_superhost"], "number_of_reviews_ltm"].agg(["count", "mean", "median", "std"]),
    "superhosts" : listings.loc[listings["host_is_superhost"], "number_of_reviews_ltm"].agg(["count", "mean", "median", "std"]),
}).round(2)

super_vs_other

,other,superhosts
count,3837.00,737.00
mean,5.76,22.08
median,1.00,16.00
std,15.18,20.21
